# 06 — RAG pipeline

Exercises the live pipeline end to end. All logic lives in `puks_rag.py` at the
repo root — this notebook only calls it, so the notebook and the deployed app
can never disagree.

    Query ─ text-embedding-3-large (3072d) over all chunks ─┐
                                                            ├─ RRF ─ Cohere rerank ─ gpt-5
    Query ─ BM25 over all chunks, independently ────────────┘

Set `AZURE_AI_KEY` before running:

```bash
export AZURE_AI_KEY=$(az cognitiveservices account keys list \
  -g <resource-group> -n <foundry-resource> --query key1 -o tsv)
```

In [ ]:
import sys, os
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
try:
    from dotenv import load_dotenv
    load_dotenv(Path.cwd().parent / ".env")
except ImportError:
    pass

import puks_rag
from puks_rag import Corpus

assert os.getenv("AZURE_AI_KEY"), "AZURE_AI_KEY is not set — see the cell above."

corpus = Corpus()
print(f"index      : {corpus.index.ntotal} vectors, d={corpus.index.d}")
print(f"chunks     : {len(corpus.chunks)}")
print(f"built with : {corpus.config.get('model_name')}")
print(f"generation : {puks_rag.CHAT_DEPLOYMENT}")
print(f"reranker   : {puks_rag.RERANK_MODEL}")

## Retrieval

Inspect which retriever surfaced each chunk. `dense`, `bm25` and `exact` are independent — a chunk found only by `bm25` is one the old pipeline could never have retrieved.

In [ ]:
question = "How do I reverse a closed GRN?"

retrieved, confidence = puks_rag.retrieve_context(corpus, question, top_k=5)

print(f"top relevance: {confidence:.4f}   (threshold {puks_rag.CONFIDENCE_THRESHOLD})\n")
for rank, r in enumerate(retrieved, 1):
    found = ", ".join(l for l, k in (("dense","in_dense"),("bm25","in_bm25"),("exact","in_exact")) if r[k])
    meta  = r["metadata"]
    print(f"{rank}. [{r['relevance_score']:.4f}] {r['doc_type']:<22} via {found}")
    print(f"   source    : {meta.get('source','?')}")
    print(f"   chunk_type: {meta.get('chunk_type','?')}")
    print(f"   {r['text'][:150].strip()}...\n")

## Context assembly

`OPERATIONAL_REFERENCE` chunks now render their `structured_data` — the validate-before-update queries and UPDATE statements that the previous pipeline dropped before the prompt was built.

In [ ]:
context_text, has_schema, has_operational = puks_rag.build_context_text(retrieved)

print(f"schema chunks     : {has_schema}")
print(f"operational chunks: {has_operational}")
print(f"context chars     : {len(context_text):,}  (~{len(context_text)//4:,} tokens)")
print("─" * 78)
print(context_text[:2500])

## Prompt

In [ ]:
intent = puks_rag.classify_query(question)
prompt = puks_rag.build_prompt(question, retrieved, "(No prior conversation)", intent)

print(intent)
print("─" * 78)
print(prompt[:1200])
print("\n...\n")
print(prompt[-900:])

## Generation

`gpt-5` is a reasoning model — `max_completion_tokens`, no `temperature`.

In [ ]:
result = puks_rag.answer(corpus, question)

print(f"refused    : {result['refused']}")
print(f"confidence : {result['confidence']:.4f}")
print("─" * 78)
print(result["answer"])

## Refusal check

An off-corpus question should fall below the threshold and refuse rather than guess.

In [ ]:
off_topic = puks_rag.answer(corpus, "What is the capital of Mongolia?")
print(f"refused    : {off_topic['refused']}")
print(f"confidence : {off_topic['confidence']:.4f}")
print(off_topic["answer"])